<div id='top'>包括以下操作：</div>
<li><a href='#1'>[:, None] 和 [None, :]</a></li>
<li><a href='#2'>torch.meshgrid</a></li>
<li><a href='#3'></a></li>
<li><a href='#4'></a></li>
<li><a href='#5'></a></li>
<li><a href='#6'></a></li>

<div style='color:skyblue; font-size:24px' id='1'>[:, None] 和 [None, :]</div>

<a href='#top'>▲ Top</a>

[:, None] 和 [None, :] 是两种常用的维度扩展操作，用于在张量的指定位置插入大小为 1 的新维度。

In [ ]:
import torch
aspect_ratios=torch.tensor([0.5, 1, 2]) 
scales = torch.tensor([32, 64]) # 形状 2    
h_ratios = torch.sqrt(aspect_ratios) # 计算高比例（h/w的平方根）
print(h_ratios)

w_ratios = 1.0 / h_ratios # 形状 3
print('w_ratios==', w_ratios)
ws = (w_ratios[:, None] * scales[None, :]).view(-1) # [:, None] 形状变成 (3,1) ； [None, :] 形状变成 (1,2)
hs = (h_ratios[:, None] * scales[None, :]).view(-1)

print('w_ratios[:, None]==', w_ratios[:, None])
print('scales[None, :]==', scales[None, :])
print(w_ratios[:, None] * scales[None, :])
print(ws)

'''
    torch.stack 是 PyTorch 中用于 合并多个张量（Tensor） 的函数，但它与 torch.cat 或 torch.concat 有本质区别。
    torch.stack 的核心行为是沿着一个新维度将多个张量堆叠起来，而不是简单地在现有维度上拼接。
'''
base_anchors = torch.stack([-ws, -hs, ws, hs], dim=1) / 2

print('base_anchors==')
print(base_anchors)

tensor([0.7071, 1.0000, 1.4142])
w_ratios== tensor([1.4142, 1.0000, 0.7071])
w_ratios[:, None]== tensor([[1.4142],
        [1.0000],
        [0.7071]])
scales[None, :]== tensor([[32, 64]])
tensor([[45.2548, 90.5097],
        [32.0000, 64.0000],
        [22.6274, 45.2548]])
tensor([45.2548, 90.5097, 32.0000, 64.0000, 22.6274, 45.2548])
base_anchors==
tensor([[-22.6274, -11.3137,  22.6274,  11.3137],
        [-45.2548, -22.6274,  45.2548,  22.6274],
        [-16.0000, -16.0000,  16.0000,  16.0000],
        [-32.0000, -32.0000,  32.0000,  32.0000],
        [-11.3137, -22.6274,  11.3137,  22.6274],
        [-22.6274, -45.2548,  22.6274,  45.2548]])


<div style='color:skyblue; font-size:24px' id='2'>torch.meshgrid</div>

<a href='#top'>▲ Top</a>

torch.meshgrid 是 PyTorch 中的一个函数，用于生成网格坐标矩阵。它的核心作用是将输入的多个一维坐标序列（例如行坐标和列坐标）组合成多维网格坐标矩阵，覆盖所有可能的坐标组合。

在目标检测的锚框生成中，shifts 张量中的 4个数值并不直接代表左上角（x_min, y_min）和右下角（x_max, y_max）的偏移，而是表示锚框中心点在原图上的坐标。

假设：

shifts 的某行为 [1,4,1,4]。

某基础锚框为 [-10, -5, 10, 5]（相对于中心点的偏移）。

最终锚框坐标为：

x_min = 1 + (-10) = -9</br>
y_min = 4 + (-5) = -1</br>
x_max = 1 + 10 = 11</br>
y_max = 4 + 5 = 9</br>

即该锚框在原图上的坐标为 (-9, -1, 11, 9)。

In [ ]:
import torch

x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5])

shift_x, shift_y = torch.meshgrid(x, y)
print("shift_x:\n", shift_x)
print("shift_y:\n", shift_y)

shift_x = shift_x.reshape(-1)
print("shift_x.reshape(-1):\n", shift_x.reshape(-1))

shift_y = shift_y.reshape(-1)
print("shift_y.reshape(-1):\n", shift_y.reshape(-1))

shifts = torch.stack([shift_x, shift_y, shift_x, shift_y], dim=1)
print("shifts:\n", shifts)

'''
    每个点的4个数值是后续计算锚框位置的基础。
    例如，若基础锚框的坐标为 (dx1, dy1, dx2, dy2)，则最终锚框坐标为：
    (x_center + dx1, y_center + dy1, x_center + dx2, y_center + dy2)
'''

shift_x:
 tensor([[1, 1],
        [2, 2],
        [3, 3]])
shift_y:
 tensor([[4, 5],
        [4, 5],
        [4, 5]])
shift_x.reshape(-1):
 tensor([1, 1, 2, 2, 3, 3])
shift_y.reshape(-1):
 tensor([4, 5, 4, 5, 4, 5])
shifts:
 tensor([[1, 4, 1, 4],
        [1, 5, 1, 5],
        [2, 4, 2, 4],
        [2, 5, 2, 5],
        [3, 4, 3, 4],
        [3, 5, 3, 5]])
